# HAM10000 classifier baseline (C0 / C1) — Colab runner

This notebook has **3 phases** (see the outline on the left):

- **Phase 1 · Setup** — GPU, Drive, paths, deps. Run once per session.
- **Phase 2 · Smoke tests** — cheap checks that prove data + training work,
  *before* you spend an hour on the real run.
- **Phase 3 · Real run** — the actual C0/C1 baselines + the results table.

**Before you start:** upload the whole `ddpm-derm-augmentation` folder to
Drive. It already contains `data/` (10k images + `manifests/`), so that is the
only upload. Then run top to bottom; you only ever edit **one cell** (1.3).

# Phase 1 · Setup
Run these five cells once each time you open the notebook.

## 1.1 GPU check
Runtime → Change runtime type → **T4 GPU** first, then run this.

In [ ]:
!nvidia-smi

## 1.2 Mount Google Drive
So checkpoints/results are written to Drive and survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.3 Paths — **EDIT THIS CELL**
Set `PROJECT_DIR` to where you uploaded the folder. `DATA_DIR` and
`OUTPUTS_DIR` derive from it automatically. The asserts fail loudly if a path
is wrong, so you can never silently run on nothing.

In [ ]:
import os
from pathlib import Path

# ==== EDIT THIS ONE LINE TO MATCH YOUR DRIVE ====
PROJECT_DIR = '/content/drive/MyDrive/ddpm-derm-augmentation'
# ================================================
DATA_DIR    = PROJECT_DIR + '/data'      # data lives inside the project
OUTPUTS_DIR = PROJECT_DIR + '/outputs'   # on Drive => checkpoints survive disconnects

os.environ['DDPM_DERM_DATA_DIR'] = DATA_DIR
os.environ['DDPM_DERM_OUTPUTS_DIR'] = OUTPUTS_DIR

assert Path(PROJECT_DIR, 'src', 'ddpm_derm', 'config.py').is_file(), \
    f'PROJECT_DIR wrong: no src/ddpm_derm/config.py under {PROJECT_DIR}'
assert Path(DATA_DIR, 'manifests', 'class_to_idx.json').is_file(), \
    f'DATA_DIR wrong: no manifests/class_to_idx.json under {DATA_DIR}'
print('paths OK')
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR    =', DATA_DIR)
print('OUTPUTS_DIR =', OUTPUTS_DIR)

## 1.4 (optional) Copy data to local disk for speed
Reading 10k files off Drive during training is slow. Copy the data folder to
Colab's fast local disk once and train from there. Local disk is wiped on
disconnect, so re-run this after a reconnect (checkpoints stay safe on Drive).
Leave it commented if you just want to get going.

In [ ]:
# !mkdir -p /content/data && cp -r "{PROJECT_DIR}/data/." /content/data/
# DATA_DIR = '/content/data'
# os.environ['DDPM_DERM_DATA_DIR'] = DATA_DIR
# assert Path(DATA_DIR, 'manifests', 'class_to_idx.json').is_file(), 'local copy layout wrong'
# print('using local data at', DATA_DIR)

## 1.5 Install light deps
torch / torchvision are already on Colab; we only add pandas + pillow.

In [ ]:
!pip install -q pandas pillow

# Phase 2 · Smoke tests
Two cheap checks. Do **both** before Phase 3 — they take under a minute and
catch a broken path / dataloader / training loop before you commit real time.

## 2.1 Data smoke test (torch-free)
Checks counts vs `split_summary`, that images open, that the fixed split has
**no lesion/image leakage**, C0/C1/C4 frame construction and the synthetic
validate/publish fixtures. Must print `60 passed, 0 failed`.

In [ ]:
!cd "{PROJECT_DIR}" && python scripts/smoke_test.py

## 2.2 Tiny training smoke — proves the loop + checkpoint/resume
1 epoch on 200 images. **Run this cell twice:**

- **1st run** prints `[start] fresh run ...`, then `[epoch 01/01] ...`,
  `... saved best.pt`, then `[test] ...`.
- **2nd run** prints `[resume] found last.pt ...` and `[skip] already trained`,
  then jumps straight to `[test]` — that proves resume reads the checkpoint.

It writes to a throwaway `outputs/_smoke/` folder, **isolated from the real
runs** (so this 200-image warm-up never contaminates your Phase-3 baseline).
Delete `_smoke/` anytime; 3.0 does it for you.

In [ ]:
!cd "{PROJECT_DIR}/src" && python -m ddpm_derm.train_classifier --variant C0 --seed 0 --epochs 1 --limit 200 --resume --output-dir "{OUTPUTS_DIR}/_smoke"

# Phase 3 · Real run
The actual baselines and the results table. This is the part that takes time.

## 3.0 (optional) Clean slate
Because `--resume` is always on, a stale checkpoint from an earlier attempt
would be *continued* instead of restarted, and a stale `results_*.json` would
be *averaged in* by 3.2. Run this once to wipe prior baseline checkpoints +
results and start fresh. **Destructive; uncomment to run.** Only touches
`outputs/`, never `data/`. (New here? Nothing to delete — it just prints that.)

In [ ]:
# import shutil
# for sub in ('classifier', '_smoke'):
#     p = Path(OUTPUTS_DIR, sub)
#     if p.exists():
#         shutil.rmtree(p); print('removed', p)
#     else:
#         print('nothing to remove at', p)

## 3.1 Baseline definitions (Stage 1 completed — do not re-run)

**Status: the Stage-1 baseline (C0 + C1 target=500, seeds 0–2) completed on
2026-07-11**; its results live in `outputs/classifier/`. This cell now only
defines `run()` / `SEEDS` / `EPOCHS` (reused by the matched-585 section 3.4);
the original Stage-1 loop is kept **commented out** for provenance. Re-running
it would `--resume` the old checkpoints — leave it alone unless you
intentionally want to reproduce Stage 1.

**What you'll see per run** (all live, one `[epoch]` line per epoch):
```
[run] variant=C1 seed=0 epochs=20 img=128 bs=32 lr=0.0003 device=cuda
[start] fresh run (no --resume) from epoch 1        # or [resume] ... after a disconnect
[ckpt] saving to .../C1_seed0 (last.pt every epoch, best.pt on improvement)
[epoch 01/20] loss=1.23 val_df_f1=0.12 val_macro_f1=0.41 (22s)
[epoch 02/20] loss=0.98 val_df_f1=0.25 val_macro_f1=0.52 (22s)  <- new best, saved best.pt
...
[test] variant=C1 seed=0 df_f1=0.42 macro_f1=0.71 acc=0.83
```
`--resume` is always passed, so re-running after a disconnect continues
instead of restarting. Watch the `NN/20` epoch counter and the `(Ns)` time to
estimate what's left.

In [ ]:
import subprocess, os, sys

SEEDS  = [0, 1, 2]     # 3-5 for the final report
EPOCHS = 20            # identical hyperparameters for every variant (C0/C1/C4)

def run(variant, seed, extra=None):
    # NOTE: subprocess.run(...) inheriting stdout does NOT show in a Colab cell.
    # Popen + PIPE + re-printing each line streams the training log live.
    cmd = [sys.executable, '-u', '-m', 'ddpm_derm.train_classifier',
           '--variant', variant, '--seed', str(seed),
           '--epochs', str(EPOCHS), '--resume']
    if extra:
        cmd += extra
    print(f'\n===== {variant} seed={seed} =====', flush=True)
    proc = subprocess.Popen(
        cmd, cwd=f'{PROJECT_DIR}/src',
        env={**os.environ, 'PYTHONPATH': f'{PROJECT_DIR}/src'},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'{variant} seed={seed} failed (exit {proc.returncode})')

# --- Stage-1 baseline loop (C0 + C1 target=500): COMPLETED 2026-07-11.
# --- Results are in outputs/classifier/. Re-running would --resume those old
# --- checkpoints; keep commented unless intentionally reproducing Stage 1.
# DF_TARGET = 500
# for s in SEEDS:
#     run('C0', s)
#     run('C1', s, ['--df-target-count', str(DF_TARGET)])

print('run() defined. Stage-1 loop is commented out (already completed); '
      'use 3.4 for the matched-585 C1/C4 runs.')

## 3.2 Aggregate: mean ± std across seeds (Stage-1 dir)

This reads the **Stage-1** results in `outputs/classifier/results/` (C0 + the
old C1 target=500). The matched-585 C1/C4 results live in a separate base and
are aggregated in **3.5**; `aggregate_results.py` reads one directory per
call, so the two tables are printed separately.

In [ ]:
!cd "{PROJECT_DIR}" && python scripts/aggregate_results.py

## 3.3 Figures — df F1, per-class recall, training curves
Reads the `results_*.json` you just produced, writes three PNGs to
`outputs/figures/` (on Drive), and shows them inline. Pure matplotlib + numpy
on the saved metrics — **no GPU, no torch, safe to re-run anytime.** Everything
is framed as *suggestive* (tiny df, fixed split, no significance test):

1. **df F1 by variant** — the headline; bars = mean ± std, dots = each seed.
2. **Per-class test recall** — checks df improved *without* hurting other classes.
3. **Validation df F1 curves** — one line per seed; shows how noisy df is.

**Note:** this cell reads only `outputs/classifier/results/` (Stage 1). The
matched-585 C1/C4 results live in `outputs/classifier_df585/results/` and do
**not** appear here yet — combining both dirs into one C0/C1/C4 figure needs an
aggregation/figures update (reported as a known limitation).

In [ ]:
# Figures from saved results -- no torch, safe to re-run anytime.
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

RESULTS_DIR = Path(OUTPUTS_DIR) / 'classifier' / 'results'
FIG_DIR     = Path(OUTPUTS_DIR) / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

runs = {}
for p in sorted(RESULTS_DIR.glob('results_*.json')):
    d = json.loads(p.read_text())
    runs.setdefault(d['variant'], []).append(d)
assert runs, f'no results_*.json in {RESULTS_DIR} -- run 3.1 first'

variants = sorted(runs)                       # e.g. ['C0', 'C1']  (C4 slots in later)
CLASSES  = list(runs[variants[0]][0]['test_metrics']['per_class_recall'].keys())
COL = {'C0': '#0072B2', 'C1': '#E69F00', 'C4': '#009E73'}   # colour-blind safe
col = lambda v: COL.get(v, '#666666')
CAPTION = 'Suggestive only: df test n=16, fixed split, no significance test.'

def tf1(v):                                   # per-seed test df F1 for a variant
    return [r['test_metrics']['target_f1'] for r in runs[v]]

# --- Fig 1: headline df F1 by variant ---------------------------------------
fig, ax = plt.subplots(figsize=(4.8, 4.2))
x = np.arange(len(variants))
means = [float(np.mean(tf1(v))) for v in variants]
stds  = [float(np.std(tf1(v)))  for v in variants]
ax.bar(x, means, yerr=stds, capsize=6, color=[col(v) for v in variants],
       alpha=0.85, edgecolor='black', linewidth=0.6)
for i, v in enumerate(variants):
    ys = tf1(v)
    ax.scatter(np.full(len(ys), x[i]), ys, color='black', s=22, zorder=3)
    ax.text(x[i], means[i] + stds[i] + 0.03, f'{means[i]:.3f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([f'{v}\n(n={len(runs[v])})' for v in variants])
ax.set_ylim(0, 1); ax.set_ylabel('test df F1 (primary metric)')
ax.set_title('df F1 by variant  (mean +/- std; dots = seeds)')
ax.text(0.5, -0.24, CAPTION, transform=ax.transAxes, ha='center', fontsize=8, color='gray')
fig.tight_layout(); fig.savefig(FIG_DIR / 'df_f1_by_variant.png', dpi=150, bbox_inches='tight')
plt.close(fig)

# --- Fig 2: per-class test recall -------------------------------------------
fig, ax = plt.subplots(figsize=(8.5, 4.2))
xc = np.arange(len(CLASSES)); w = 0.8 / len(variants)
for j, v in enumerate(variants):
    vals = np.array([[r['test_metrics']['per_class_recall'][c] for c in CLASSES]
                     for r in runs[v]])
    ax.bar(xc + j * w, vals.mean(0), w, yerr=vals.std(0), capsize=3,
           label=v, color=col(v), alpha=0.85)
ax.set_xticks(xc + w * (len(variants) - 1) / 2); ax.set_xticklabels(CLASSES)
for lbl in ax.get_xticklabels():
    if lbl.get_text() == 'df':
        lbl.set_fontweight('bold')            # highlight the target class
ax.set_ylim(0, 1); ax.set_ylabel('test recall (mean +/- std)')
ax.set_title('Per-class test recall  (did df improve without hurting others?)')
ax.legend(title='variant')
fig.tight_layout(); fig.savefig(FIG_DIR / 'per_class_recall.png', dpi=150, bbox_inches='tight')
plt.close(fig)

# --- Fig 3: validation df F1 training curves --------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for v in variants:
    for k, r in enumerate(runs[v]):
        ep = [h['epoch'] for h in r['history']]
        f1 = [h['val_df_f1'] for h in r['history']]
        ax.plot(ep, f1, color=col(v), alpha=0.55, label=v if k == 0 else None)
ax.set_xlabel('epoch'); ax.set_ylabel('val df F1'); ax.set_ylim(0, 1)
ax.set_title('Validation df F1 per seed  (note the instability; df val n=14)')
ax.legend(title='variant')
fig.tight_layout(); fig.savefig(FIG_DIR / 'val_df_f1_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig)

for name in ('df_f1_by_variant.png', 'per_class_recall.png', 'val_df_f1_curves.png'):
    display(Image(str(FIG_DIR / name)))
print('saved 3 figures ->', FIG_DIR)


## 3.4 Matched-585: C1 vs C4 — the formal comparison

C4's train frame = the fixed train split + the **published epoch-100 synthetic
set** (85 real + 500 generated = **585 df**). C1 is re-run matched to the same
total (`--df-target-count 585`, duplicating real train df only), so the two
differ **only in where the extra df rows come from**. Both use the exact same
`run()` hyperparameters as Stage 1 (epochs/img/batch/lr) — run cell **3.1**
first for the definitions.

Results go to a **new base** `outputs/classifier_df585/` so the Stage-1
results in `outputs/classifier/` (C0 + old C1 target=500) are never touched.
**C0 is not re-run** — it is reused from Stage 1 (the raw train split is
identical either way).

The cell refuses to start until `_READY.json` exists in the published
synthetic dir, i.e. the DDPM notebook's 3.2 sample → validate → publish flow
succeeded. `--resume` is passed as usual, so a disconnect resumes instead of
restarting.

In [ ]:
import json
from pathlib import Path

DF585_BASE   = OUTPUTS_DIR + '/classifier_df585'   # new base; Stage-1 outputs/classifier stays untouched
SYN_DIR      = Path(OUTPUTS_DIR) / 'synthetic_df' / 'epoch0100_seed0'
SYN_MANIFEST = SYN_DIR / 'synthetic_df.csv'
READY        = SYN_DIR / '_READY.json'

assert READY.is_file(), (
    f'formal synthetic set is not published yet: missing {READY}\n'
    'run the DDPM notebook section 3.2 (sample -> validate -> publish) first')
print('_READY.json OK, published:', json.loads(READY.read_text())['published_utc'])

DF_TARGET_585 = 585          # 85 real + 500 generated; C1 matched to C4's df total
SEEDS_585     = [0, 1, 2]

for s in SEEDS_585:
    run('C1', s, ['--df-target-count', str(DF_TARGET_585),
                  '--output-dir', DF585_BASE])
    run('C4', s, ['--df-target-count', str(DF_TARGET_585),
                  '--generated-manifest', str(SYN_MANIFEST),
                  '--output-dir', DF585_BASE])

## 3.5 Aggregate the matched-585 results

Two separate tables (one directory per `aggregate_results.py` call): the
matched-585 C1/C4 runs, then the Stage-1 C0 (+ old C1 target=500) for
reference. The matched comparison is **C1@585 vs C4@585** with C0 from Stage 1
as the imbalanced baseline. Merging all three into a single table/figure needs
an aggregation update (known limitation, reported).

In [ ]:
print('===== matched-585 (C1@585 vs C4@585) =====')
!cd "{PROJECT_DIR}" && python scripts/aggregate_results.py --results-dir "{DF585_BASE}/results"
print('\n===== Stage-1 reference (C0 + old C1 target=500) =====')
!cd "{PROJECT_DIR}" && python scripts/aggregate_results.py --results-dir "{OUTPUTS_DIR}/classifier/results"

---
**Artifacts (already on Drive under `outputs/`):**
- results JSON: `outputs/classifier/results/`
- checkpoints: `outputs/classifier/checkpoints/<variant>_seed<seed>/best.pt`
- figures: `outputs/figures/*.png` (from 3.3)

Because `OUTPUTS_DIR` is on Drive, these are saved as you go — nothing extra to
upload. Keep `best.pt` for the later FastAPI/HF deployment stage.